# Web Scraping — Resultados Únicos Saber 11
**Fuente:** [datos.gov.co — dataset kgxf-xxbe](https://www.datos.gov.co/Educaci-n/Resultados-nicos-Saber-11/kgxf-xxbe/about_data)

**Tecnología:** PySpark + Socrata SODA API  
**Objetivo:** Extraer datos del portal de datos abiertos del gobierno colombiano y generar visualizaciones analíticas.

---
## ¿Por qué SODA API y no BeautifulSoup?

El portal `datos.gov.co` utiliza **Socrata**, una plataforma que expone los datasets mediante la **SODA (Socrata Open Data API)**. Esto significa que los datos **no están en HTML** sino en un endpoint REST accesible directamente como JSON/CSV, sin necesidad de parsear HTML:

```
GET https://www.datos.gov.co/resource/kgxf-xxbe.json?$limit=50000&$offset=0
```

Esto es **más robusto** que scraping HTML (no se rompe si cambia el diseño de la página) y **más eficiente** (paginación nativa, filtros del lado del servidor).


## 1. Instalación de dependencias

In [0]:
# Ejecutar si las librerías no están instaladas
!pip install pyspark requests matplotlib pandas

## 2. Imports

In [0]:
import requests
import json

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

print("✅ Imports listos")

## 3. Inicializar SparkSession

In [0]:
spark = (
    SparkSession.builder
    .appName("WebScraper_Saber11")
    .config("spark.sql.repl.eagerEval.enabled", "true")
    .getOrCreate()
)
print("SparkSession iniciada")

## 4. Web Scraping — Socrata SODA API

El endpoint base del dataset es:
```
https://www.datos.gov.co/resource/kgxf-xxbe.json
```

Parámetros clave de la SODA API:
| Parámetro | Descripción |
|-----------|------------|
| `$limit`  | Registros por página (máx. 50 000) |
| `$offset` | Desplazamiento para paginación |
| `$order`  | Columna de ordenamiento (estabiliza la paginación) |
| `$where`  | Filtros tipo SQL |
| `$select` | Proyección de columnas |


In [0]:
BASE_URL  = "https://www.datos.gov.co/resource/kgxf-xxbe.json"
PAGE_SIZE = 50_000     # máximo por request en Socrata
MAX_ROWS  = 500_000    # ajustar a None para descargar todo el dataset

def fetch_all_rows(base_url: str, page_size: int = 50_000,
                   max_rows: int | None = None) -> list[dict]:
    """
    Descarga todos los registros del dataset mediante paginación SODA.
    Devuelve una lista de diccionarios (JSON).
    """
    all_rows = []
    offset   = 0

    print(f"Conectando a: {base_url}")
    print(f"Página: {page_size:,} registros | Límite: {max_rows if max_rows else 'sin límite'}\n")

    while True:
        params = {
            "$limit" : page_size,
            "$offset": offset,
            "$order" : ":id",   # orden estable para paginación
        }
        response = requests.get(base_url, params=params, timeout=60)
        response.raise_for_status()

        page = response.json()
        if not page:
            print("\nNo hay más registros.")
            break

        all_rows.extend(page)
        offset += len(page)
        print(f"Descargados: {len(all_rows):,}", end="\r")

        if max_rows and len(all_rows) >= max_rows:
            all_rows = all_rows[:max_rows]
            print(f"\nLímite de {max_rows:,} alcanzado.")
            break

        if len(page) < page_size:   # última página
            break

    print(f"\nTotal: {len(all_rows):,} registros\n")
    return all_rows

# ── Ejecutar scraping ──
raw_data = fetch_all_rows(BASE_URL, page_size=PAGE_SIZE, max_rows=MAX_ROWS)

## 5. Crear DataFrame Spark

In [0]:
# Crear DataFrame Spark directamente desde lista de dicts (sin sparkContext)
df = spark.createDataFrame(raw_data)

print("Schema inferido:")
df.printSchema()
print(f"\n   Filas    : {df.count():,}")
print(f"   Columnas : {len(df.columns)}")

## 6. Limpieza y transformación

In [0]:
# Columnas de puntajes presentes en el dataset
puntaje_cols = [
    "punt_matematicas", "punt_sociales_ciudadanas", "punt_c_naturales",
    "punt_lectura_critica", "punt_ingles", "punt_global"
]

# Convertir puntajes de string → double
for col in puntaje_cols:
    if col in df.columns:
        df = df.withColumn(col, F.col(col).cast(DoubleType()))

# Eliminar filas sin puntaje global
df_clean = df.dropna(subset=["punt_global"])

# Estandarizar columnas categóricas
for col_str in ["cole_naturaleza", "cole_jornada", "estu_genero",
                "estu_estrato_vivienda", "cole_depto_ubicacion"]:
    if col_str in df_clean.columns:
        df_clean = df_clean.withColumn(col_str, F.upper(F.trim(F.col(col_str))))

df_clean.createOrReplaceTempView("saber11")
print(f"Vista SQL 'saber11' registrada | {df_clean.count():,} filas limpias")

## 7. Análisis con Spark SQL

In [0]:
# ── Análisis 1: Promedio puntaje global por Departamento ──
df_dept = spark.sql("""
    SELECT cole_depto_ubicacion                AS departamento,
           COUNT(*)                            AS n_estudiantes,
           ROUND(AVG(punt_global), 2)          AS prom_global,
           ROUND(AVG(punt_matematicas), 2)     AS prom_mates,
           ROUND(AVG(punt_lectura_critica), 2) AS prom_lectura
    FROM   saber11
    WHERE  cole_depto_ubicacion IS NOT NULL
    GROUP  BY cole_depto_ubicacion
    ORDER  BY prom_global DESC
""")
df_dept.show(20, truncate=False)

In [0]:
# ── Análisis 2: Oficial vs No Oficial ──
df_naturaleza = spark.sql("""
    SELECT cole_naturaleza,
           COUNT(*)                       AS n_estudiantes,
           ROUND(AVG(punt_global), 2)     AS prom_global,
           ROUND(AVG(punt_matematicas),2) AS prom_mates,
           ROUND(AVG(punt_ingles), 2)     AS prom_ingles
    FROM   saber11
    WHERE  cole_naturaleza IS NOT NULL
    GROUP  BY cole_naturaleza
    ORDER  BY prom_global DESC
""")
df_naturaleza.show(truncate=False)

In [0]:
# ── Análisis 3: Puntaje global por Estrato de Vivienda ──
df_estrato = spark.sql("""
    SELECT fami_estratovivienda      AS estrato,
           COUNT(*)                  AS n_estudiantes,
           ROUND(AVG(punt_global), 2) AS prom_global
    FROM   saber11
    WHERE  fami_estratovivienda IS NOT NULL
      AND  fami_estratovivienda NOT IN ('SIN ESTRATO','SE IGNORA','')
    GROUP  BY fami_estratovivienda
    ORDER  BY estrato
""")
df_estrato.show(truncate=False)

In [0]:
# ── Análisis 4: Distribución por período / año ──
df_periodo = spark.sql("""
    SELECT periodo,
           COUNT(*) AS n_estudiantes
    FROM   saber11
    WHERE  periodo IS NOT NULL
    GROUP  BY periodo
    ORDER  BY periodo
""")
df_periodo.show(30, truncate=False)

## 8. Visualizaciones

In [0]:
C1, C2, C3 = "#1a73e8", "#ea4335", "#34a853"

fig, axes = plt.subplots(2, 2, figsize=(18, 13))
fig.suptitle(
    "Resultados Saber 11 — Análisis Exploratorio\n"
    "Fuente: datos.gov.co | Dataset kgxf-xxbe",
    fontsize=14, fontweight="bold", y=0.98
)

pd_dept       = df_dept.limit(15).toPandas()
pd_naturaleza = df_naturaleza.toPandas()
pd_estrato    = df_estrato.toPandas()
pd_periodo    = df_periodo.toPandas()

# — Gráfico 1: Top 15 departamentos —
ax1 = axes[0, 0]
bars = ax1.barh(pd_dept["departamento"], pd_dept["prom_global"],
                color=C1, edgecolor="white")
ax1.set_xlabel("Puntaje Global Promedio")
ax1.set_title("Top 15 Departamentos — Puntaje Global")
ax1.invert_yaxis()
for bar, val in zip(bars, pd_dept["prom_global"]):
    ax1.text(bar.get_width() - 1, bar.get_y() + bar.get_height()/2,
             f"{val:.1f}", va="center", ha="right",
             color="white", fontsize=8, fontweight="bold")

# — Gráfico 2: Oficial vs No oficial —
ax2 = axes[0, 1]
categorias = ["prom_global", "prom_mates", "prom_ingles"]
etiquetas  = ["Global", "Matemáticas", "Inglés"]
x, w = range(len(categorias)), 0.35
naturalezas = pd_naturaleza["cole_naturaleza"].tolist()
colors_nat  = [C1, C2, "#fbbc04"]
for i, (nat, color) in enumerate(zip(naturalezas, colors_nat)):
    fila = pd_naturaleza[pd_naturaleza["cole_naturaleza"] == nat]
    vals = [fila[c].values[0] if len(fila) else 0 for c in categorias]
    offset = (i - len(naturalezas)/2 + 0.5) * w
    ax2.bar([xi + offset for xi in x], vals, w,
            label=nat.title(), color=color, edgecolor="white")
ax2.set_xticks(list(x)); ax2.set_xticklabels(etiquetas)
ax2.set_ylabel("Puntaje Promedio")
ax2.set_title("Puntajes por Naturaleza del Colegio")
ax2.legend()

# — Gráfico 3: Estrato vs puntaje —
ax3 = axes[1, 0]
pde = pd_estrato.sort_values("estrato")
ax3.plot(pde["estrato"], pde["prom_global"],
         marker="o", color=C3, linewidth=2.5, markersize=8,
         markerfacecolor="white", markeredgewidth=2)
for _, row in pde.iterrows():
    ax3.annotate(f"{row['prom_global']:.1f}", (row["estrato"], row["prom_global"]),
                 textcoords="offset points", xytext=(0, 10),
                 ha="center", fontsize=9, color=C3, fontweight="bold")
ax3.set_xlabel("Estrato de Vivienda")
ax3.set_ylabel("Puntaje Global Promedio")
ax3.set_title("Estrato de Vivienda vs Puntaje Global")
ax3.grid(axis="y", alpha=0.3)

# — Gráfico 4: Tendencia por período —
ax4 = axes[1, 1]
pd_periodo["periodo"] = pd_periodo["periodo"].astype(str)
ax4.bar(pd_periodo["periodo"], pd_periodo["n_estudiantes"],
        color=C1, edgecolor="white", alpha=0.85)
ax4.set_xlabel("Período"); ax4.set_ylabel("N° Estudiantes")
ax4.set_title("Estudiantes por Período")
ax4.tick_params(axis="x", rotation=45)
ax4.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.tight_layout()
plt.savefig("resultados_saber11.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico guardado: resultados_saber11.png")

## 9. Guardar resultados en CSV

## 10. Cerrar SparkSession

In [0]:
spark.stop()
print("SparkSession cerrada. Proceso finalizado.")